# Trino AI Chat Interface

Chat interface powered by GitHub Copilot and the Trino MCP Server.

In [ ]:
import os
import json
import subprocess
import sys
import time
import requests
import urllib3
import openai
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Disable SSL warnings for MITM proxy
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## Configuration

Set your Trino credentials. GitHub Copilot authentication is handled via device code flow in the next cell.

In [ ]:
# Trino OAuth Configuration (only set if not already provided by environment)
os.environ.setdefault("TRINO_HOST", "sql.example.com:443")
os.environ.setdefault("AUTH_MODE", "client_credentials")
os.environ.setdefault("CLIENT_ID", "your-client-id-here")
os.environ.setdefault("CLIENT_SECRET", "your-client-secret-here")
os.environ.setdefault("TOKEN_ENDPOINT", "https://oauth.example.com/token")
os.environ["KEYRING_CRYPTFILE_PASSWORD"] = "set_me_please"

print(f"✅ Trino configuration set")
print(f"  Trino Host: {os.environ['TRINO_HOST']}")

## GitHub Copilot Authentication

Authenticates via GitHub device code flow (same as VS Code / IntelliJ). Tokens are saved locally for reuse.

In [ ]:
# VS Code's official OAuth client ID - same one used by all Copilot extensions
COPILOT_CLIENT_ID = "Iv1.b507a08c87ecfe98"
COPILOT_TOKEN_FILE = os.path.join(os.getcwd(), ".copilot_github_token.json")

VSCODE_HEADERS = {
    "Editor-Version": "vscode/1.96.0",
    "Editor-Plugin-Version": "copilot/1.246.0",
    "User-Agent": "GithubCopilot/1.246.0",
    "Accept": "application/json",
}

def load_copilot_session():
    if os.path.exists(COPILOT_TOKEN_FILE):
        try:
            with open(COPILOT_TOKEN_FILE, "r") as f:
                return json.load(f)
        except Exception:
            pass
    return {}

def save_copilot_session(github_token, copilot_token, expires_at, api_url):
    with open(COPILOT_TOKEN_FILE, "w") as f:
        json.dump({
            "github_token": github_token,
            "copilot_token": copilot_token,
            "copilot_expires_at": expires_at,
            "copilot_api_url": api_url,
        }, f)

def copilot_device_auth():
    """GitHub device code OAuth flow - same as VS Code / IntelliJ."""
    r = requests.post(
        "https://github.com/login/device/code",
        headers={"Accept": "application/json"},
        data={"client_id": COPILOT_CLIENT_ID, "scope": "read:user"},
        verify=False,
    )
    data = r.json()
    if "user_code" not in data:
        raise RuntimeError(f"Device code request failed: {data}")

    user_code = data["user_code"]
    device_code = data["device_code"]
    interval = data.get("interval", 5)
    expires_in = data.get("expires_in", 900)

    print(f"\n  ┌─────────────────────────────────┐")
    print(f"  │                                 │")
    print(f"  │     Code:  {user_code}             │")
    print(f"  │                                 │")
    print(f"  └─────────────────────────────────┘")
    print(f"\n  Open: https://github.com/login/device")
    print(f"  Enter the code above and authorize.\n")
    print(f"  Waiting for authorization... (timeout: {expires_in}s)")

    start_time = time.time()
    while time.time() - start_time < expires_in:
        time.sleep(interval)
        r = requests.post(
            "https://github.com/login/oauth/access_token",
            headers={"Accept": "application/json"},
            data={
                "client_id": COPILOT_CLIENT_ID,
                "device_code": device_code,
                "grant_type": "urn:ietf:params:oauth:grant-type:device_code",
            },
            verify=False,
        )
        result = r.json()
        error = result.get("error")
        if error == "authorization_pending":
            continue
        elif error == "slow_down":
            interval += 5
            continue
        elif error in ("expired_token", "access_denied"):
            raise RuntimeError(f"Authorization failed: {error}")
        elif "access_token" in result:
            print(f"\n✅ Authorization successful!")
            return result["access_token"]
        else:
            raise RuntimeError(f"Unknown response: {result}")

    raise RuntimeError("Authorization timeout")

def get_copilot_token(github_token):
    """Exchange GitHub OAuth token for a short-lived Copilot session token."""
    r = requests.get(
        "https://api.github.com/copilot_internal/v2/token",
        headers={**VSCODE_HEADERS, "Authorization": f"token {github_token}"},
        verify=False,
    )
    if r.status_code != 200:
        raise RuntimeError(
            f"Copilot token exchange failed (HTTP {r.status_code}): {r.text[:300]}\n"
            "Check your Copilot subscription: https://github.com/settings/copilot"
        )
    data = r.json()
    return (
        data["token"],
        data.get("expires_at", 0),
        data.get("endpoints", {}).get("api", "https://api.githubcopilot.com"),
    )

# --- Authenticate ---
session = load_copilot_session()
github_token = session.get("github_token")

# Check if cached Copilot token is still valid (with 60s buffer)
if github_token and session.get("copilot_token") and time.time() < session.get("copilot_expires_at", 0) - 60:
    copilot_token = session["copilot_token"]
    copilot_expires_at = session["copilot_expires_at"]
    copilot_api_url = session["copilot_api_url"]
    print(f"✅ Using cached Copilot token (expires at {time.strftime('%H:%M:%S', time.localtime(copilot_expires_at))})")
elif github_token:
    print("Refreshing Copilot token...")
    try:
        copilot_token, copilot_expires_at, copilot_api_url = get_copilot_token(github_token)
        save_copilot_session(github_token, copilot_token, copilot_expires_at, copilot_api_url)
        print(f"✅ Copilot token refreshed (expires at {time.strftime('%H:%M:%S', time.localtime(copilot_expires_at))})")
    except Exception:
        print("GitHub token invalid, re-authenticating...")
        github_token = None

if github_token is None:
    github_token = copilot_device_auth()
    copilot_token, copilot_expires_at, copilot_api_url = get_copilot_token(github_token)
    save_copilot_session(github_token, copilot_token, copilot_expires_at, copilot_api_url)
    print(f"✅ Copilot token acquired (expires at {time.strftime('%H:%M:%S', time.localtime(copilot_expires_at))})")

print(f"   API endpoint: {copilot_api_url}")

## MCP Client

In [ ]:
class MCPClient:
    def __init__(self):
        self.process = None
        self.message_id = 0
        
    def start(self):
        self.process = subprocess.Popen(
            [sys.executable, "-m", "mcp_server_trino"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
            env=os.environ.copy()
        )
        
        print("Initializing MCP server...")
        init_result = self.send_request("initialize", {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "trino-jupyter", "version": "1.0.0"}
        })
        print(f"✅ MCP Server started: {init_result['serverInfo']['name']}")
        
        self.send_notification("notifications/initialized")
        
    def send_notification(self, method, params=None):
        notification = {"jsonrpc": "2.0", "method": method}
        if params:
            notification["params"] = params
        self.process.stdin.write(json.dumps(notification) + "\n")
        self.process.stdin.flush()
        
    def send_request(self, method, params=None):
        self.message_id += 1
        request = {"jsonrpc": "2.0", "id": self.message_id, "method": method}
        if params is not None:
            request["params"] = params
            
        self.process.stdin.write(json.dumps(request) + "\n")
        self.process.stdin.flush()
        
        response_line = self.process.stdout.readline()
        if not response_line:
            stderr = self.process.stderr.read()
            raise RuntimeError(f"MCP server died: {stderr}")
            
        response = json.loads(response_line)
        if "error" in response:
            raise RuntimeError(f"MCP Error: {response['error']}")
        return response.get("result")
        
    def list_tools(self):
        return self.send_request("tools/list", {})
        
    def list_resources(self):
        return self.send_request("resources/list", {})
        
    def call_tool(self, tool_name, arguments):
        return self.send_request("tools/call", {"name": tool_name, "arguments": arguments})
        
    def stop(self):
        if self.process:
            self.process.terminate()
            self.process.wait(timeout=5)
            print("✅ MCP Server stopped")

mcp_client = MCPClient()

## Start MCP Server

In [ ]:
mcp_client.start()

tools = mcp_client.list_tools()
print(f"\nAvailable tools: {[t['name'] for t in tools['tools']]}")

resources = mcp_client.list_resources()
print(f"Available tables: {len(resources.get('resources', []))} found")

## GitHub Copilot Chat Client

In [ ]:
import httpx

# Clear SSL environment variables that interfere with httpx
os.environ.pop('SSL_CERT_FILE', None)
os.environ.pop('SSL_CERT_DIR', None)
os.environ.pop('REQUESTS_CA_BUNDLE', None)
os.environ.pop('CURL_CA_BUNDLE', None)

# Create OpenAI client pointing to Copilot API
copilot_http_client = httpx.Client(
    verify=False,
    timeout=60.0,
    headers={
        "Copilot-Integration-Id": "vscode-chat",
        "Editor-Version": "vscode/1.96.0",
        "Editor-Plugin-Version": "copilot/1.246.0",
        "User-Agent": "GithubCopilot/1.246.0",
    },
)

client = OpenAI(
    base_url=copilot_api_url,
    api_key=copilot_token,
    http_client=copilot_http_client,
)

def refresh_copilot_client():
    """Refresh Copilot token if it's about to expire (~30 min lifetime)."""
    global client, copilot_token, copilot_expires_at
    if time.time() > copilot_expires_at - 60:
        print("Refreshing Copilot token...")
        copilot_token, copilot_expires_at, _ = get_copilot_token(github_token)
        save_copilot_session(github_token, copilot_token, copilot_expires_at, copilot_api_url)
        client = OpenAI(
            base_url=copilot_api_url,
            api_key=copilot_token,
            http_client=copilot_http_client,
        )
        print("✅ Token refreshed")

# Fetch available models - use cached file if it exists
MODELS_CACHE_FILE = os.path.join(os.getcwd(), ".copilot_models.json")
available_models = []

if os.path.exists(MODELS_CACHE_FILE):
    try:
        with open(MODELS_CACHE_FILE, "r") as f:
            available_models = json.load(f)
        if available_models:
            print(f"✅ Loaded {len(available_models)} models from cache")
    except Exception:
        available_models = []

if not available_models:
    print("Fetching models from Copilot API...")
    models_resp = requests.get(
        f"{copilot_api_url}/models",
        headers={
            "Authorization": f"Bearer {copilot_token}",
            **VSCODE_HEADERS,
            "Copilot-Integration-Id": "vscode-chat",
        },
        verify=False,
    )
    if models_resp.status_code == 200:
        models_data = models_resp.json()
        for m in models_data.get("data", models_data if isinstance(models_data, list) else []):
            model_id = m.get("id", m) if isinstance(m, dict) else m
            available_models.append(model_id)
        available_models.sort()
        if available_models:
            with open(MODELS_CACHE_FILE, "w") as f:
                json.dump(available_models, f, indent=2)
            print(f"✅ Fetched and cached {len(available_models)} models")

if not available_models:
    available_models = ["gpt-4o", "gpt-4o-mini", "gpt-4.1", "claude-sonnet-4", "o3-mini"]

MODEL = "gpt-4o" if "gpt-4o" in available_models else available_models[0]

print(f"   Available models ({len(available_models)}): {', '.join(available_models)}")
print(f"   Default model: {MODEL}")

## Chat Interface

In [ ]:
import re

chat_history = []
output_area = widgets.Output()
input_box = widgets.Textarea(placeholder='Ask about your Trino data...', layout=widgets.Layout(width='100%', height='100px'))
send_button = widgets.Button(description='Send', button_style='primary')
clear_button = widgets.Button(description='Clear', button_style='warning')

# Filter out non-chat models (embeddings, routers)
chat_models = [m for m in available_models if not m.startswith(("text-embedding", "accounts/"))]
model_dropdown = widgets.Dropdown(
    options=chat_models,
    value=MODEL if MODEL in chat_models else chat_models[0],
    description='Model:',
    layout=widgets.Layout(width='300px'),
)

TOOLS_DEF = [{"type": "function", "function": {"name": "execute_sql", "description": "Execute SQL on Trino", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}]

SYSTEM_PROMPT = {
    "role": "system",
    "content": (
        "You have access to a Trino SQL database. When you need to query data, "
        "write SQL inside a ```sql code block. The system will execute it and return results. "
        "You can then analyze the results and answer the user's question. "
        "Always execute queries rather than guessing. Write one query at a time."
    )
}

def get_msg_field(msg, field):
    if isinstance(msg, dict):
        return msg.get(field)
    return getattr(msg, field, None)

def extract_sql_blocks(text):
    """Extract SQL from ```sql code blocks."""
    return re.findall(r'```sql\s*\n(.*?)```', text, re.DOTALL | re.IGNORECASE)

def call_chat_with_tools(selected_model, messages):
    """Try tool-calling API. Returns (response, True) or (None, False)."""
    try:
        return client.chat.completions.create(
            model=selected_model, messages=messages,
            tools=TOOLS_DEF, tool_choice="auto"
        ), True
    except openai.BadRequestError:
        pass
    try:
        return client.chat.completions.create(
            model=selected_model, messages=messages,
            tools=TOOLS_DEF
        ), True
    except openai.BadRequestError:
        pass
    return None, False

def render_chat():
    with output_area:
        clear_output()
        for msg in chat_history:
            role = get_msg_field(msg, "role")
            content = get_msg_field(msg, "content")
            if not content:
                continue
            if role == "user" and not content.startswith("Query result"):
                display(HTML(f'<div style="background-color: #e3f2fd; padding: 10px; margin: 5px; border-radius: 5px;"><strong>You:</strong> {content}</div>'))
            elif role == "assistant":
                formatted = re.sub(
                    r'```sql\s*\n(.*?)```',
                    r'<pre style="background:#222;color:#0f0;padding:8px;border-radius:4px;overflow-x:auto"><code>\1</code></pre>',
                    content, flags=re.DOTALL
                )
                display(HTML(f'<div style="background-color: #f1f8e9; padding: 10px; margin: 5px; border-radius: 5px;"><strong>AI:</strong> {formatted}</div>'))
            elif role == "user" and content.startswith("Query result"):
                display(HTML(f'<div style="background-color: #e8eaf6; padding: 10px; margin: 5px; border-radius: 5px; font-family: monospace; font-size: 0.85em; white-space: pre-wrap;">{content}</div>'))

def send_message(b):
    user_message = input_box.value.strip()
    if not user_message:
        return
    input_box.value = ''
    selected_model = model_dropdown.value
    chat_history.append({"role": "user", "content": user_message})
    
    with output_area:
        display(HTML(f'<div style="background-color: #e3f2fd; padding: 10px; margin: 5px; border-radius: 5px;"><strong>You:</strong> {user_message}</div>'))
        display(HTML(f'<div style="background-color: #fff3e0; padding: 10px; margin: 5px; border-radius: 5px;"><em>Thinking ({selected_model})...</em></div>'))
        
    try:
        refresh_copilot_client()
        messages = [SYSTEM_PROMPT] + chat_history
        
        # Try tool-calling API first (works for GPT models)
        response, used_tools = call_chat_with_tools(selected_model, messages)
        
        if used_tools:
            assistant_message = response.choices[0].message
            while assistant_message.tool_calls:
                chat_history.append(assistant_message)
                for tool_call in assistant_message.tool_calls:
                    result = mcp_client.call_tool(tool_call.function.name, json.loads(tool_call.function.arguments))
                    result_text = "\n".join([item['text'] for item in result['content'] if item['type'] == 'text'])
                    chat_history.append({"role": "tool", "tool_call_id": tool_call.id, "name": tool_call.function.name, "content": result_text})
                refresh_copilot_client()
                messages = [SYSTEM_PROMPT] + chat_history
                response, _ = call_chat_with_tools(selected_model, messages)
                assistant_message = response.choices[0].message
            chat_history.append({"role": "assistant", "content": assistant_message.content})
        else:
            # SQL-in-code-block mode (Claude, Gemini, and other models)
            for _ in range(5):
                response = client.chat.completions.create(model=selected_model, messages=messages)
                reply = response.choices[0].message.content or ""
                chat_history.append({"role": "assistant", "content": reply})
                
                sql_blocks = extract_sql_blocks(reply)
                if not sql_blocks:
                    break
                
                for sql in sql_blocks:
                    sql = sql.strip()
                    try:
                        result = mcp_client.call_tool("execute_sql", {"query": sql})
                        result_text = "\n".join([item['text'] for item in result['content'] if item['type'] == 'text'])
                    except Exception as e:
                        result_text = f"Error: {e}"
                    chat_history.append({"role": "user", "content": f"Query result for: {sql}\n\n{result_text}"})
                
                messages = [SYSTEM_PROMPT] + chat_history
        
        render_chat()
        
    except Exception as e:
        import traceback
        error_details = traceback.format_exc()
        with output_area:
            clear_output()
            display(HTML(f'<div style="background-color: #ffebee; padding: 10px; margin: 5px; border-radius: 5px;"><strong>Error:</strong> {str(e)}<br><pre style="white-space: pre-wrap;">{error_details}</pre></div>'))

def clear_chat(b):
    global chat_history
    chat_history = []
    with output_area:
        clear_output()

send_button.on_click(send_message)
clear_button.on_click(clear_chat)

display(HTML('<h2>Trino AI Chat</h2>'))
display(model_dropdown)
display(output_area)
display(input_box)
display(widgets.HBox([send_button, clear_button]))

## Cleanup

In [ ]:
mcp_client.stop()